# 🚀 4-Hour Hands-On Tutorial: LangChain with Google Gemini

**Platform:** Google Colab  
**Language:** Python  
**LLM:** Google Gemini through `langchain-google-genai`  
**Primary model:** `gemini-3.5-flash`  
**Level:** Beginner → Intermediate  
**Duration:** Approximately 4 hours

---

## What you will build

By the end of this notebook, you will have hands-on experience with:

1. Connecting LangChain to Google Gemini.
2. Sending prompts to Gemini through LangChain.
3. Using prompt templates.
4. Building chains with LCEL (`|` operator).
5. Parsing text and structured outputs.
6. Building a simple conversational assistant.
7. Creating custom tools.
8. Building a Retrieval-Augmented Generation (RAG) pipeline.
9. Completing a mini-project: **AI Customer Support Knowledge Assistant**.

> **Important:** Never hard-code your API key into notebooks that you intend to share publicly.


## ⏱️ Suggested 4-Hour Agenda

| Time | Module | Hands-on Outcome |
|---|---|---|
| 0:00–0:25 | Module 1 — Setup and Gemini basics | Connect Gemini to LangChain |
| 0:25–1:05 | Module 2 — Prompt templates and chains | Build reusable LCEL chains |
| 1:05–1:35 | Module 3 — Output parsing and structured data | Extract validated business data |
| 1:35–2:05 | Module 4 — Conversation and message history | Build a context-aware chatbot |
| 2:05–2:35 | Module 5 — Tools and tool calling | Give the model callable functions |
| 2:35–3:20 | Module 6 — Embeddings and RAG | Ask questions over private documents |
| 3:20–4:00 | Module 7 — Mini-project | Build a customer-support knowledge assistant |


# Module 1 — Setup and Your First Gemini Call

## 1.1 What is LangChain?

LangChain is an application framework for building AI systems that combine language models with prompts, tools, retrieval systems, structured outputs, and application logic.

A useful mental model is:

**User Question → Prompt → Model → Optional Tools/Retrieval → Structured Response**

In this tutorial, Google Gemini is the language model, while LangChain is the orchestration layer.


## 1.2 Install the required packages

Run the following cell in Google Colab.

The notebook installs:

- `langchain`
- `langchain-google-genai`
- `langchain-community`
- `langchain-text-splitters`
- `pydantic`

After installation, Colab may occasionally ask you to restart the runtime when dependency versions change.


In [43]:
%pip install -qU langchain langchain-google-genai langchain-community langchain-text-splitters pydantic

## 1.3 Add your Google Gemini API key securely

Create a Gemini API key from Google AI Studio, then enter it below.

The input is hidden by `getpass`, so the key is not displayed in the notebook output.


In [44]:
import os
import getpass

if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google Gemini API key: ")

print("✅ GOOGLE_API_KEY is configured for this Colab session.")

✅ GOOGLE_API_KEY is configured for this Colab session.


## 1.4 Initialize the Gemini chat model

For this tutorial we use `gemini-3.5-flash`.

`temperature` controls response randomness. A higher value tends to produce more varied answers.


In [45]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=1.0,
    max_retries=2,
)

print("✅ Gemini model initialized.")

✅ Gemini model initialized.


## 1.5 Your first model call

In [46]:
response = llm.invoke("Explain LangChain in simple terms for a beginner in three sentences.")
print(response.content)

[{'type': 'text', 'text': 'LangChain is a framework that makes it easier to connect powerful AI models, like ChatGPT, to your own data and external tools. It acts as a bridge, allowing the AI to "read" your private files, search the internet, or perform specific actions rather than just relying on its pre-trained knowledge. Essentially, it helps you build smarter applications by chaining together different components to create complex, automated workflows.', 'extras': {'signature': 'EjQKMgERTTIPo9ZnwMX43grhUQcpAmA930Wf6DBLaXhOK17IReCUFS1DbcKDLnVBnk4cW0e/'}}]


### Exercise 1

Change the prompt and ask Gemini to explain one of these topics in simple terms:

- RAG
- AI agents
- Vector databases
- Prompt engineering

Try adding constraints such as: *Use an analogy* or *Explain in exactly five bullet points*.


In [47]:
# TODO: Replace the prompt with your own question.

exercise_response = llm.invoke(
    "Explain Retrieval-Augmented Generation using a library analogy."
)
print(exercise_response.content)

[{'type': 'text', 'text': 'To understand **Retrieval-Augmented Generation (RAG)**, imagine you are a student taking a final exam.\n\n### 1. The Traditional LLM (The "Memorizer")\nWithout RAG, an AI model is like a student who has spent years reading every book in the world and then had those books taken away. During the exam, they must rely entirely on their memory.\n\n*   **The Problem:** If the student studied five years ago, they don’t know about events that happened yesterday. If they memorized the wrong information, they might "hallucinate" (make things up) because they are trying to guess the answer from fading memories.\n\n### 2. The RAG Model (The "Open-Book Exam")\nRAG changes the process. Instead of relying solely on memory, the AI is allowed to perform an **open-book exam**. Here is how the analogy breaks down:\n\n#### Step 1: The Retrieval (The Librarian)\nWhen you ask the AI a question, it doesn\'t answer immediately. Instead, it acts like a **librarian**. It rushes into a

# Module 2 — Prompt Templates and LCEL Chains

## 2.1 Why use prompt templates?

Hard-coded prompts become difficult to maintain when an application has many users or changing inputs.

A prompt template separates:

- fixed instructions;
- dynamic variables;
- reusable application logic.


In [48]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert teacher. Explain concepts clearly for a beginner."),
    ("human", "Explain {topic} using a real-world example.")
])

formatted_prompt = prompt.invoke({"topic": "vector embeddings"})
print(formatted_prompt)

messages=[SystemMessage(content='You are an expert teacher. Explain concepts clearly for a beginner.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain vector embeddings using a real-world example.', additional_kwargs={}, response_metadata={})]


## 2.2 Build your first LCEL chain

LangChain Expression Language, commonly called **LCEL**, lets us connect components with the pipe operator:

`prompt | model | parser`


In [49]:
from langchain_core.output_parsers import StrOutputParser

text_parser = StrOutputParser()

basic_chain = prompt | llm | text_parser

result = basic_chain.invoke({"topic": "AI agents"})
print(result)

To understand an AI agent, it helps to first understand the difference between a **standard AI** (like ChatGPT) and an **AI agent.**

*   **Standard AI** is like a library assistant: you ask a question, and it provides an answer based on what it knows. It stops there.
*   **An AI Agent** is like an executive assistant: you give it a **goal**, and it figures out the steps, uses tools, and completes tasks to achieve that goal.

---

### The Real-World Example: Planning a Business Trip

Imagine you have a human personal assistant named Alex. If you tell Alex, *"I need to go to London for a conference next Tuesday,"* Alex doesn't just give you a list of flight prices. Alex **acts** for you.

Here is how an AI agent performs that same task:

#### 1. Perception (Understanding the Goal)
The agent hears your request: "London, next Tuesday, for a conference." It identifies the objective and breaks it into smaller sub-tasks:
*   Find the conference dates.
*   Find flights that arrive before the 

## 2.3 Build a business-oriented chain

Imagine that a manager wants a concise explanation of an AI use case.


In [50]:
business_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a business AI consultant. Return a concise and practical explanation."
    ),
    (
        "human",
        """Analyze the following AI use case:

Use case: {use_case}
Industry: {industry}

Explain:
1. Business problem
2. How AI can help
3. Required data
4. Expected business value
5. One implementation risk
"""
    )
])

business_chain = business_prompt | llm | StrOutputParser()

business_result = business_chain.invoke({
    "use_case": "Predict which customers are likely to churn",
    "industry": "Telecommunications"
})

print(business_result)

### AI Use Case: Predictive Churn Analysis (Telecommunications)

**1. Business Problem**
High customer acquisition costs make retention critical. Telcos suffer from "revenue leakage" when customers switch to competitors, often without warning, due to pricing, poor service, or better offers elsewhere.

**2. How AI Can Help**
Machine learning models (e.g., Random Forest, XGBoost) analyze historical patterns to assign a "churn probability score" to each customer. This allows the business to transition from reactive support to proactive, targeted retention efforts (e.g., personalized discounts or service upgrades) before the customer decides to leave.

**3. Required Data**
*   **Usage Data:** Call volume, data consumption, and roaming frequency.
*   **Account Data:** Contract length, billing history, and payment delays.
*   **Interaction Data:** Customer support ticket history, chat logs, and complaint frequency.
*   **Demographic Data:** Age, location, and device type.
*   **External Fact

## 2.4 Chain composition

A major LangChain idea is that the output of one component can become the input of another.

Below, the first chain creates a summary and the second chain creates quiz questions from that summary.


In [51]:
summary_prompt = ChatPromptTemplate.from_template(
    "Explain {topic} for a beginner in approximately 150 words."
)

quiz_prompt = ChatPromptTemplate.from_template(
    """Based only on the following explanation, create five multiple-choice questions.
Each question must have four options and provide the correct answer.

Explanation:
{explanation}
"""
)

summary_chain = summary_prompt | llm | StrOutputParser()
quiz_chain = quiz_prompt | llm | StrOutputParser()

explanation = summary_chain.invoke({"topic": "supervised machine learning"})
quiz = quiz_chain.invoke({"explanation": explanation})

print("=== EXPLANATION ===")
print(explanation)

print("\n=== QUIZ ===")
print(quiz)

=== EXPLANATION ===
Supervised machine learning is like teaching a student with the help of an answer key. 

In this process, you provide a computer with a dataset that contains both the input (data) and the correct output (labels). Think of it as showing the computer thousands of photos labeled "cat" or "dog." By analyzing these examples, the algorithm learns to identify the patterns and relationships that distinguish a cat from a dog.

Once the training is complete, the computer can apply what it has learned to new, unseen data. If you show it a photo it hasn’t seen before, it will use those learned patterns to predict whether it is a cat or a dog.

Common uses include predicting house prices based on features like size or identifying spam emails. Essentially, supervised learning turns historical data into a predictive tool, enabling computers to make accurate guesses about the future.

=== QUIZ ===
Based on the explanation provided, here are five multiple-choice questions:

**1. How

### Exercise 2

Build a chain that accepts:

- a job role;
- a technical topic;
- the learner's experience level.

The chain should produce a personalized 7-day learning plan.

A starter solution is provided below. Modify it and experiment.


In [52]:
learning_prompt = ChatPromptTemplate.from_template(
    """You are an expert technical trainer.

Create a 7-day learning plan.

Job role: {role}
Topic: {topic}
Experience level: {level}

For each day provide:
- Learning objective
- Topics
- One hands-on task
- Expected outcome

Keep the plan practical.
"""
)

learning_chain = learning_prompt | llm | StrOutputParser()

plan = learning_chain.invoke({
    "role": "SAP Developer",
    "topic": "Generative AI and LangChain",
    "level": "Beginner"
})

print(plan)

This 7-day intensive plan is designed for an SAP Developer (ABAP/CAP/RAP background) to bridge the gap into Generative AI using LangChain.

### Prerequisites
*   **Environment:** Python (VS Code), OpenAI API Key, and an SAP BTP Trial account (for context).
*   **Mindset:** Focus on how LLMs interface with structured enterprise data.

---

### Day 1: The Foundations of LLMs & LangChain
*   **Objective:** Understand how Large Language Models work and set up the development environment.
*   **Topics:** Intro to LLMs, Tokenization, Prompt Engineering basics, Setting up Python/LangChain.
*   **Hands-on Task:** Install LangChain and call the OpenAI API to generate a summary of a mock "SAP Sales Order" JSON object.
*   **Outcome:** You can successfully run a Python script that interacts with an LLM.

### Day 2: Prompt Templates & Chain Logic
*   **Objective:** Master the art of structured input to get predictable outputs.
*   **Topics:** PromptTemplates, LCEL (LangChain Expression Language), 

# Module 3 — Structured Output

Free-form text is useful for humans, but applications often need predictable fields.

For example, a support application may need:

```text
category
priority
sentiment
summary
recommended_team
```

LangChain can ask supported models to return output matching a Pydantic schema.


In [53]:
from pydantic import BaseModel, Field
from typing import Literal

class SupportTicket(BaseModel):
    category: Literal[
        "Billing",
        "Technical",
        "Account",
        "Delivery",
        "Other"
    ] = Field(description="Primary ticket category")

    priority: Literal["High", "Medium", "Low"] = Field(
        description="Urgency of the issue"
    )

    sentiment: Literal["Positive", "Neutral", "Negative"] = Field(
        description="Customer sentiment"
    )

    summary: str = Field(
        description="Short summary of the customer problem"
    )

    recommended_team: str = Field(
        description="Team that should handle the issue"
    )

structured_llm = llm.with_structured_output(SupportTicket)

ticket_text = """
I have been charged twice for my subscription this month.
I contacted support yesterday but have not received a reply.
Please refund the duplicate charge immediately.
"""

ticket = structured_llm.invoke(
    f"Analyze this customer support ticket:\n\n{ticket_text}"
)

ticket

SupportTicket(category='Billing', priority='High', sentiment='Negative', summary='Customer charged twice for subscription and has not received a response to previous inquiry.', recommended_team='Billing Support')

## 3.1 Convert the validated object to a dictionary

In [54]:
ticket.model_dump()

{'category': 'Billing',
 'priority': 'High',
 'sentiment': 'Negative',
 'summary': 'Customer charged twice for subscription and has not received a response to previous inquiry.',
 'recommended_team': 'Billing Support'}

### Exercise 3 — Employee feedback analyzer

Create a schema containing:

- `main_topic`
- `sentiment`
- `urgency`
- `summary`
- `action_required`

Then analyze the feedback below.


In [55]:
class EmployeeFeedback(BaseModel):
    main_topic: str
    sentiment: Literal["Positive", "Neutral", "Negative"]
    urgency: Literal["High", "Medium", "Low"]
    summary: str
    action_required: bool

feedback_analyzer = llm.with_structured_output(EmployeeFeedback)

feedback = """
The new internal learning portal is useful, but search is extremely slow.
I need to complete mandatory training by tomorrow and the portal repeatedly times out.
"""

feedback_result = feedback_analyzer.invoke(
    f"Analyze the following employee feedback:\n\n{feedback}"
)

feedback_result

EmployeeFeedback(main_topic='Internal learning portal performance issues', sentiment='Negative', urgency='High', summary='The employee finds the portal useful but is experiencing severe technical issues with search functionality and timeouts, which is preventing them from meeting a critical deadline for mandatory training.', action_required=True)

# Module 4 — Conversation and Message History

A basic model call does not automatically remember previous Python calls.

To build a conversation, we need to pass prior messages back to the model.

LangChain provides standard message classes such as:

- `HumanMessage`
- `AIMessage`
- `SystemMessage`


In [56]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser

# Using a parser ensures we get a string even if the model returns a complex list
parser = StrOutputParser()
chain = llm | parser

messages = [
    SystemMessage(
        content="You are a helpful AI tutor. Keep answers concise and beginner-friendly."
    ),
    HumanMessage(content="My name is Rohan and I am learning LangChain."),
]

first_reply_text = chain.invoke(messages)
print(f"First reply: {first_reply_text}")

# We still append the AI Message object to history for context
messages.append(AIMessage(content=first_reply_text))
messages.append(HumanMessage(content="What is my name and what am I learning?"))

second_reply_text = chain.invoke(messages)
print("\nSecond reply: " + second_reply_text)

First reply: Hi Rohan! Welcome to the world of LangChain. It’s a powerful tool for building applications with Large Language Models (LLMs).

To get you started, just remember that LangChain is essentially a framework designed to "chain" different components together—like connecting a prompt to an AI model, or giving that model access to your own files.

**Here are the 3 core concepts to start with:**

1.  **Models:** The "brains" (like GPT-4 or Claude).
2.  **Prompts:** How you structure instructions for the model.
3.  **Chains:** Connecting the components so the output of one step becomes the input for the next.

Do you have a specific project in mind, or would you like a simple example to run first?

Second reply: Your name is **Rohan**, and you are learning **LangChain**!


## 4.1 Build a simple conversational loop

Run the following cell and chat until you type `exit`.

> In a production system, conversation history would usually be persisted in a database or managed through a dedicated state layer rather than kept only in a Python list.


In [57]:
from langchain_core.messages import HumanMessage, SystemMessage

chat_history = [
    SystemMessage(
        content="You are a patient LangChain tutor. Give concise, practical answers."
    )
]

print("Type 'exit' to stop.\n")

while True:
    user_input = input("You: ").strip()

    if user_input.lower() == "exit":
        print("Chat ended.")
        break

    chat_history.append(HumanMessage(content=user_input))

    assistant_response = llm.invoke(chat_history)
    chat_history.append(assistant_response)

    print(f"Gemini: {assistant_response.content}\n")

Type 'exit' to stop.

You: exit
Chat ended.


# Module 5 — Tools and Tool Calling

A **tool** is a function that an AI model can request to use.

Examples:

- get weather;
- query a database;
- calculate loan payments;
- search product inventory;
- create a support ticket.

We will first create simple local tools.


In [58]:
from langchain_core.tools import tool

@tool
def calculate_discount(price: float, discount_percent: float) -> float:
    """Calculate the final price after applying a percentage discount."""
    return round(price * (1 - discount_percent / 100), 2)

@tool
def get_order_status(order_id: str) -> str:
    """Get the status of an order from a simulated order database."""
    mock_orders = {
        "ORD-1001": "Shipped",
        "ORD-1002": "Processing",
        "ORD-1003": "Delivered",
    }
    return mock_orders.get(order_id, "Order not found")

print(calculate_discount.invoke({"price": 1000, "discount_percent": 15}))
print(get_order_status.invoke({"order_id": "ORD-1001"}))

850.0
Shipped


## 5.1 Bind tools to Gemini

Binding tools tells the model what functions are available.

The model may return a tool-call request instead of final natural-language text.


In [59]:
tools = [calculate_discount, get_order_status]
llm_with_tools = llm.bind_tools(tools)

tool_response = llm_with_tools.invoke(
    "What is the status of order ORD-1002?"
)

print("Content:", tool_response.content)
print("Tool calls:", tool_response.tool_calls)

Content: []
Tool calls: [{'name': 'get_order_status', 'args': {'order_id': 'ORD-1002'}, 'id': 'KVGiPxDY', 'type': 'tool_call'}]


## 5.2 Execute requested tool calls manually

This demonstrates the essential mechanics behind tool-using agents.


In [60]:
tool_map = {tool.name: tool for tool in tools}

user_question = "What is the final price of a ₹2500 product after a 12% discount?"
ai_message = llm_with_tools.invoke(user_question)

if ai_message.tool_calls:
    for tool_call in ai_message.tool_calls:
        selected_tool = tool_map[tool_call["name"]]
        tool_result = selected_tool.invoke(tool_call["args"])

        print("Tool selected:", tool_call["name"])
        print("Arguments:", tool_call["args"])
        print("Tool result:", tool_result)
else:
    print(ai_message.content)

Tool selected: calculate_discount
Arguments: {'price': 2500, 'discount_percent': 12}
Tool result: 2200.0


### Exercise 4 — Create your own business tool

Create a tool called `calculate_simple_interest` that accepts:

- principal;
- annual interest rate;
- number of years.

Formula:

`Simple Interest = Principal × Rate × Time / 100`

Then bind it to Gemini and ask a natural-language question.


In [61]:
@tool
def calculate_simple_interest(
    principal: float,
    annual_rate: float,
    years: float
) -> float:
    """Calculate simple interest from principal, annual rate percentage, and time in years."""
    return round(principal * annual_rate * years / 100, 2)

finance_llm = llm.bind_tools([calculate_simple_interest])

finance_response = finance_llm.invoke(
    "Calculate the simple interest on ₹50,000 at 7.5% per year for 3 years."
)

print(finance_response.tool_calls)

[{'name': 'calculate_simple_interest', 'args': {'principal': 50000, 'years': 3, 'annual_rate': 7.5}, 'id': '1ca6hgYi', 'type': 'tool_call'}]


# Module 6 — Embeddings and RAG

## 6.1 What is RAG?

Retrieval-Augmented Generation gives an LLM relevant information from an external knowledge source before it answers.

Typical flow:

**Documents → Split into chunks → Embeddings → Vector store → Retrieve relevant chunks → Gemini generates grounded answer**

This is useful when the model needs access to:

- company policies;
- product documentation;
- contracts;
- manuals;
- SAP documentation;
- internal knowledge bases.


## 6.2 Create sample private documents

For a real application, these could come from PDF files, websites, databases, SharePoint, SAP systems, or other repositories.

For this hands-on tutorial, we use three policy documents directly in Python.


In [62]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="""
        Refund Policy:
        Customers may request a full refund within 30 calendar days of purchase.
        After 30 days, refunds are not normally available unless the product is defective.
        Approved refunds are processed within 5 to 7 business days.
        """,
        metadata={"source": "refund_policy"}
    ),
    Document(
        page_content="""
        Shipping Policy:
        Standard delivery takes 3 to 5 business days.
        Express delivery takes 1 to 2 business days.
        Orders above ₹2,000 qualify for free standard delivery.
        """,
        metadata={"source": "shipping_policy"}
    ),
    Document(
        page_content="""
        Premium Support Policy:
        Premium customers receive 24/7 email and chat support.
        Standard customers receive support Monday to Friday from 9 AM to 6 PM.
        Critical premium incidents have a target initial response time of one hour.
        """,
        metadata={"source": "support_policy"}
    ),
]

print(f"Created {len(documents)} documents.")

Created 3 documents.


## 6.3 Split documents into chunks

Smaller chunks improve retrieval granularity.


In [63]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print(f"Number of chunks: {len(chunks)}")
for i, chunk in enumerate(chunks, start=1):
    print(f"\n--- Chunk {i} ---")
    print(chunk.page_content.strip())

Number of chunks: 3

--- Chunk 1 ---
Refund Policy:
        Customers may request a full refund within 30 calendar days of purchase.
        After 30 days, refunds are not normally available unless the product is defective.
        Approved refunds are processed within 5 to 7 business days.

--- Chunk 2 ---
Shipping Policy:
        Standard delivery takes 3 to 5 business days.
        Express delivery takes 1 to 2 business days.
        Orders above ₹2,000 qualify for free standard delivery.

--- Chunk 3 ---
Premium Support Policy:
        Premium customers receive 24/7 email and chat support.
        Standard customers receive support Monday to Friday from 9 AM to 6 PM.
        Critical premium incidents have a target initial response time of one hour.


## 6.4 Create Gemini embeddings and an in-memory vector store

This notebook uses an in-memory vector store to keep setup simple.

For larger production applications, you may use a persistent vector database.


In [64]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview",
    output_dimensionality=768,
)

vectorstore = InMemoryVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("✅ Vector store created.")

✅ Vector store created.


## 6.5 Test retrieval before generation

Good RAG systems inspect retrieval quality separately from final answer quality.


In [65]:
question = "How long does a refund take after approval?"

retrieved_docs = retriever.invoke(question)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"--- Retrieved document {i} ---")
    print(doc.page_content.strip())
    print("Source:", doc.metadata.get("source"))
    print()

--- Retrieved document 1 ---
Refund Policy:
        Customers may request a full refund within 30 calendar days of purchase.
        After 30 days, refunds are not normally available unless the product is defective.
        Approved refunds are processed within 5 to 7 business days.
Source: refund_policy

--- Retrieved document 2 ---
Shipping Policy:
        Standard delivery takes 3 to 5 business days.
        Express delivery takes 1 to 2 business days.
        Orders above ₹2,000 qualify for free standard delivery.
Source: shipping_policy

--- Retrieved document 3 ---
Premium Support Policy:
        Premium customers receive 24/7 email and chat support.
        Standard customers receive support Monday to Friday from 9 AM to 6 PM.
        Critical premium incidents have a target initial response time of one hour.
Source: support_policy



## 6.6 Build the RAG chain

The model is explicitly instructed to answer only from retrieved context and admit when the context is insufficient.


In [66]:
rag_prompt = ChatPromptTemplate.from_template(
    """You are a customer-support knowledge assistant.

Answer the user's question using only the context below.

Rules:
1. Do not invent information.
2. If the answer is not present in the context, say:
   "I do not have enough information in the provided knowledge base."
3. Keep the answer concise.
4. Mention the relevant policy when possible.

Context:
{context}

Question:
{question}
"""
)

def format_docs(docs):
    return "\n\n".join(
        f"Source: {doc.metadata.get('source')}\n{doc.page_content}"
        for doc in docs
    )

def ask_rag(question: str) -> str:
    docs = retriever.invoke(question)
    context = format_docs(docs)
    return (rag_prompt | llm | StrOutputParser()).invoke({
        "context": context,
        "question": question
    })

print(ask_rag("Can I get a refund 20 days after purchase?"))

Yes, you may request a full refund within 30 calendar days of purchase according to the Refund Policy.


## 6.7 Test multiple RAG questions

In [72]:
import time

questions = [
    "How many days does standard shipping take?",
    "Who receives 24/7 support?",
    "What happens if I ask for a refund after 45 days?",
    "What is the company's office address?"
]

for q in questions:
    try:
        print(f"QUESTION: {q}")
        print("ANSWER:", ask_rag(q))
        print("-" * 80)
        # Small delay to respect free-tier rate limits
        time.sleep(2)
    except Exception as e:
        if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):
            print(f"Rate limit hit. Waiting 10 seconds before continuing...")
            time.sleep(10)
            # Retry once after the long wait
            try:
                print("ANSWER (Retry):", ask_rag(q))
            except Exception as retry_e:
                print(f"Still failing: {retry_e}")
        else:
            print(f"Error processing question '{q}': {e}")

QUESTION: How many days does standard shipping take?
ANSWER: According to the Shipping Policy, standard delivery takes 3 to 5 business days.
--------------------------------------------------------------------------------
QUESTION: Who receives 24/7 support?
ANSWER: Premium customers receive 24/7 email and chat support, according to the Premium Support Policy.
--------------------------------------------------------------------------------
QUESTION: What happens if I ask for a refund after 45 days?
ANSWER: According to the Refund Policy, refunds are not normally available after 30 days unless the product is defective.
--------------------------------------------------------------------------------
QUESTION: What is the company's office address?
ANSWER: I do not have enough information in the provided knowledge base.
--------------------------------------------------------------------------------


# Module 7 — Mini-Project: AI Customer Support Knowledge Assistant

## Problem statement

Build an AI assistant that:

1. receives a customer question;
2. retrieves relevant company-policy information;
3. generates a grounded answer;
4. classifies the request;
5. returns a structured result that another application could consume.

The final output will contain:

- `answer`
- `category`
- `priority`
- `needs_human_agent`
- `source_policies`


In [73]:
from typing import List

class CustomerSupportResponse(BaseModel):
    answer: str = Field(description="Grounded answer to the customer")
    category: Literal["Refund", "Shipping", "Support", "Other"]
    priority: Literal["High", "Medium", "Low"]
    needs_human_agent: bool
    source_policies: List[str] = Field(
        description="Knowledge-base sources used for the response"
    )

support_structured_llm = llm.with_structured_output(CustomerSupportResponse)

## 7.1 Create the final application function

In [74]:
final_support_prompt = ChatPromptTemplate.from_template(
    """You are an AI customer support assistant.

Use only the supplied knowledge-base context.

Customer question:
{question}

Knowledge-base context:
{context}

Instructions:
- Provide a concise grounded answer.
- Never invent a policy.
- Classify the request.
- Set needs_human_agent to true when:
  - the knowledge base is insufficient,
  - the request requires an exception,
  - there is an unresolved urgent complaint.
- Include only sources that appear in the supplied context.
"""
)

def customer_support_assistant(question: str) -> CustomerSupportResponse:
    retrieved_docs = retriever.invoke(question)
    context = format_docs(retrieved_docs)

    chain = final_support_prompt | support_structured_llm

    return chain.invoke({
        "question": question,
        "context": context
    })

## 7.2 Test the complete assistant

In [75]:
test_questions = [
    "I bought the product 15 days ago. Can I return it for a full refund?",
    "My standard delivery has taken 10 business days and has not arrived.",
    "I am a premium customer. Is chat support available at midnight?",
    "Can I exchange my purchase for a different color?"
]

for question in test_questions:
    print(f"\nCUSTOMER: {question}")
    result = customer_support_assistant(question)
    print(result.model_dump_json(indent=2))
    print("=" * 100)


CUSTOMER: I bought the product 15 days ago. Can I return it for a full refund?
{
  "answer": "Yes, you are eligible for a full refund since you purchased the product 15 days ago, which is within our 30-day return window. Once approved, the refund will be processed within 5 to 7 business days.",
  "category": "Refund",
  "priority": "Low",
  "needs_human_agent": false,
  "source_policies": [
    "refund_policy"
  ]
}

CUSTOMER: My standard delivery has taken 10 business days and has not arrived.
{
  "answer": "Standard delivery typically takes 3 to 5 business days. Since your order has taken 10 business days and has not arrived, I have escalated this to a human agent to investigate the status of your shipment.",
  "category": "Shipping",
  "priority": "High",
  "needs_human_agent": true,
  "source_policies": [
    "shipping_policy"
  ]
}

CUSTOMER: I am a premium customer. Is chat support available at midnight?
{
  "answer": "Yes, as a premium customer, you have access to 24/7 chat sup

# Optional Challenge — Turn the Project into a Simple Interactive App

Use this loop to interact with the support assistant directly inside Colab.


In [76]:
print("AI Customer Support Assistant")
print("Type 'exit' to stop.\n")

while True:
    question = input("Customer: ").strip()

    if question.lower() == "exit":
        print("Session ended.")
        break

    result = customer_support_assistant(question)

    print("\nAnswer:", result.answer)
    print("Category:", result.category)
    print("Priority:", result.priority)
    print("Human agent required:", result.needs_human_agent)
    print("Sources:", ", ".join(result.source_policies))
    print("-" * 80)

AI Customer Support Assistant
Type 'exit' to stop.

Customer: return the product

Answer: You may request a full refund within 30 calendar days of your purchase. Once approved, refunds are typically processed within 5 to 7 business days. Please provide your order details so we can assist you with the return process.
Category: Refund
Priority: Medium
Human agent required: True
Sources: refund_policy
--------------------------------------------------------------------------------
Customer: i want sydney sweeny

Answer: I am sorry, but I do not have any information regarding 'sydney sweeny' in our system. If you have a question about our products or services, please let me know.
Category: Other
Priority: Low
Human agent required: True
Sources: 
--------------------------------------------------------------------------------
Customer: it is a product

Answer: Could you please provide more details regarding your inquiry about the product so I can better assist you?
Category: Other
Priority:

# Final Hands-On Assignment

Build a **Course Recommendation Assistant** using LangChain and Gemini.

## Requirements

Create at least five sample course documents. Each course should include:

- course name;
- skills taught;
- experience level;
- duration;
- prerequisites.

Then build a RAG application that accepts questions such as:

> I am an SAP ABAP developer with no AI experience. Which course should I take first to learn SAP Business AI?

Your system should return structured output with:

- `recommended_courses`
- `reason`
- `prerequisites`
- `learning_sequence`
- `confidence`

## Bonus requirements

1. Add a custom tool that calculates total learning hours.
2. Add conversation history.
3. Return the result as a Pydantic model.
4. Add source metadata to each recommendation.
5. Build a Streamlit front end.


### 1. Define Course Data and Create Vector Store

In [77]:
import os
import time
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

try:
    from langchain_google_genai import GoogleGenerativeAIEmbeddings
except ImportError:
    print("Package not found, re-installing...")
    !pip install -qU langchain-google-genai
    from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Explicitly fetch the API key from environment
api_key = os.getenv("GOOGLE_API_KEY")

# Initialize embeddings passing the google_api_key explicitly
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview",
    output_dimensionality=768,
    google_api_key=api_key
)

course_docs = [
    Document(
        page_content="Course: AI Essentials for ABAPers. Skills: Python basics, Prompt Engineering. Level: Beginner. Duration: 10 hours. Prerequisites: Basic programming knowledge.",
        metadata={"name": "AI Essentials for ABAPers", "duration": 10}
    ),
    Document(
        page_content="Course: SAP Business AI Fundamentals. Skills: Joule, AI Core, Generative AI Hub. Level: Beginner. Duration: 15 hours. Prerequisites: None.",
        metadata={"name": "SAP Business AI Fundamentals", "duration": 15}
    ),
    Document(
        page_content="Course: LangChain for SAP Developers. Skills: RAG, Tool use, Chain building. Level: Intermediate. Duration: 20 hours. Prerequisites: Python, AI Essentials.",
        metadata={"name": "LangChain for SAP Developers", "duration": 20}
    ),
    Document(
        page_content="Course: Advanced AI Core Orchestration. Skills: Model fine-tuning, Deployment. Level: Advanced. Duration: 30 hours. Prerequisites: SAP Business AI Fundamentals.",
        metadata={"name": "Advanced AI Core Orchestration", "duration": 30}
    ),
    Document(
        page_content="Course: Data Science for SAP Consultants. Skills: Pandas, Scikit-learn, SQL. Level: Intermediate. Duration: 25 hours. Prerequisites: SQL knowledge.",
        metadata={"name": "Data Science for SAP Consultants", "duration": 25}
    )
]

# Create the vector store for courses
course_vectorstore = InMemoryVectorStore.from_documents(
    documents=course_docs,
    embedding=embeddings,
)
course_retriever = course_vectorstore.as_retriever(search_kwargs={"k": 3})
print("✅ Course Vector Store and Retriever are ready.")

✅ Course Vector Store and Retriever are ready.


### 2. Define Structured Output and Custom Tool

In [78]:
from pydantic import BaseModel, Field
from typing import List
from langchain_core.tools import tool

class CourseRecommendation(BaseModel):
    recommended_courses: List[str] = Field(description="List of course names")
    reason: str = Field(description="Why these courses were selected based on user profile")
    prerequisites: List[str] = Field(description="List of prerequisites for the recommended path")
    learning_sequence: str = Field(description="The order in which to take the courses")
    confidence: float = Field(description="Confidence score from 0 to 1")

@tool
def calculate_total_hours(course_names: List[str]) -> int:
    """Calculate the total study hours for a list of course names."""
    total = 0
    # Mapping for tool look-up
    duration_map = {doc.metadata['name']: doc.metadata['duration'] for doc in course_docs}
    for name in course_names:
        total += duration_map.get(name, 0)
    return total

### 3. Build the Assistant Logic

In [79]:
from langchain_core.prompts import ChatPromptTemplate

course_prompt = ChatPromptTemplate.from_template(
    """You are a Course Recommendation Assistant for SAP Professionals.

User Background: {question}
Context (Available Courses):
{context}

Recommend the best courses and explain the sequence. Return structured data."""
)

def get_course_advice(user_query: str):
    docs = course_retriever.invoke(user_query)
    context = "\n".join([d.page_content for d in docs])

    # Use structured output model
    advisor_llm = llm.with_structured_output(CourseRecommendation)

    recommendation = (course_prompt | advisor_llm).invoke({
        "question": user_query,
        "context": context
    })

    # Use the tool to add information
    total_hours = calculate_total_hours.invoke({"course_names": recommendation.recommended_courses})

    return recommendation, total_hours

# --- Test Run ---
query = "I am an SAP ABAP developer with no AI experience. Which course should I take first to learn SAP Business AI?"
advice, hours = get_course_advice(query)

print(f"Recommendations: {advice.recommended_courses}")
print(f"Total Hours: {hours}")
print(f"Reasoning: {advice.reason}")
display(advice.model_dump())

Recommendations: ['SAP Business AI Fundamentals', 'AI Essentials for ABAPers']
Total Hours: 25
Reasoning: As an ABAP developer new to AI, starting with the fundamentals provides the necessary conceptual framework for SAP's AI architecture, while the AI Essentials course bridges your existing development skills with modern AI practices like Prompt Engineering.


{'recommended_courses': ['SAP Business AI Fundamentals',
  'AI Essentials for ABAPers'],
 'reason': "As an ABAP developer new to AI, starting with the fundamentals provides the necessary conceptual framework for SAP's AI architecture, while the AI Essentials course bridges your existing development skills with modern AI practices like Prompt Engineering.",
 'prerequisites': ['Basic ABAP programming knowledge'],
 'learning_sequence': 'Start with SAP Business AI Fundamentals to understand the platform landscape, then proceed to AI Essentials for ABAPers to apply these concepts through a developer-centric lens.',
 'confidence': 0.95}

### 4. Add Conversational Memory

In [80]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Global history for the session
course_chat_history = []

def get_conversational_course_advice(user_query: str):
    # 1. Retrieve context
    docs = course_retriever.invoke(user_query)
    context = "\n".join([d.page_content for d in docs])

    # 2. Build messages with history
    system_msg = SystemMessage(content=f"You are a Course Recommendation Assistant. Use the following context to help the user:\n{context}")

    current_messages = [system_msg] + course_chat_history + [HumanMessage(content=user_query)]

    # 3. Get structured output
    advisor_llm = llm.with_structured_output(CourseRecommendation)
    recommendation = advisor_llm.invoke(current_messages)

    # 4. Use tool for hours
    total_hours = calculate_total_hours.invoke({"course_names": recommendation.recommended_courses})

    # 5. Update history
    course_chat_history.append(HumanMessage(content=user_query))
    course_chat_history.append(AIMessage(content=recommendation.reason))

    return recommendation, total_hours

# --- Test Conversation ---
print("--- Round 1 ---")
rec1, h1 = get_conversational_course_advice("I am a beginner in AI but I know ABAP.")
print(f"Recommended: {rec1.recommended_courses} ({h1} hours)")

print("\n--- Round 2 (Context Aware) ---")
rec2, h2 = get_conversational_course_advice("I have a busy schedule, suggest only the shortest one from those.")
print(f"Shortest: {rec2.recommended_courses} ({h2} hours)")
print(f"Reasoning: {rec2.reason}")

--- Round 1 ---
Recommended: ['AI Essentials for ABAPers', 'SAP Business AI Fundamentals', 'LangChain for SAP Developers'] (45 hours)

--- Round 2 (Context Aware) ---
Shortest: ['AI Essentials for ABAPers'] (10 hours)
Reasoning: Since you are an ABAPer looking for the shortest entry point, this course directly leverages your existing skill set and has the lowest time commitment.


### 5. Final Verified Assistant with Source Metadata

In [81]:
from pydantic import BaseModel, Field
from typing import List

# Ensure the base class is available in this scope
class CourseRecommendation(BaseModel):
    recommended_courses: List[str] = Field(description="List of course names")
    reason: str = Field(description="Why these courses were selected based on user profile")
    prerequisites: List[str] = Field(description="List of prerequisites for the recommended path")
    learning_sequence: str = Field(description="The order in which to take the courses")
    confidence: float = Field(description="Confidence score from 0 to 1")

class EnhancedCourseRecommendation(CourseRecommendation):
    source_metadata: List[dict] = Field(description="Metadata from the retrieved course documents")

def get_final_recommendation(user_query: str):
    docs = course_retriever.invoke(user_query)
    context = "\n".join([d.page_content for d in docs])
    metadata = [d.metadata for d in docs]

    advisor_llm = llm.with_structured_output(EnhancedCourseRecommendation)

    res = (course_prompt | advisor_llm).invoke({
        "question": user_query,
        "context": context
    })

    res.source_metadata = metadata
    total_hours = calculate_total_hours.invoke({"course_names": res.recommended_courses})

    return res, total_hours

print("Running final check...")
try:
    final_advice, final_hours = get_final_recommendation("Suggest a path for an ABAPer to learn SAP AI Core.")
    print(f"Total Path Duration: {final_hours} hours")
    display(final_advice.model_dump())
except Exception as e:
    print(f"Error during execution: {e}")

Running final check...
Total Path Duration: 55 hours


{'recommended_courses': ['SAP Business AI Fundamentals',
  'AI Essentials for ABAPers',
  'Advanced AI Core Orchestration'],
 'reason': "This path starts by building conceptual knowledge of SAP's AI ecosystem, then provides the necessary programming skills (Python) required to bridge ABAP expertise with AI Core, and concludes with advanced orchestration and deployment techniques.",
 'prerequisites': ['Basic programming knowledge',
  'SAP Business AI Fundamentals'],
 'learning_sequence': '1. SAP Business AI Fundamentals, 2. AI Essentials for ABAPers, 3. Advanced AI Core Orchestration',
 'confidence': 0.95,
 'source_metadata': [{'name': 'AI Essentials for ABAPers', 'duration': 10},
  {'name': 'SAP Business AI Fundamentals', 'duration': 15},
  {'name': 'Advanced AI Core Orchestration', 'duration': 30}]}

# Common Errors and Troubleshooting

## 1. Authentication error

Check that `GOOGLE_API_KEY` is correctly configured.

```python
import os
print(bool(os.getenv("GOOGLE_API_KEY")))
```

Never print the actual key.

## 2. Model not found

Gemini model availability can change. Check the current Google Gemini model documentation and replace the model ID in the initialization cell.

## 3. Import error

Re-run the installation cell and restart the Colab runtime when necessary.

## 4. Rate-limit or quota error

Wait for quota availability, reduce repeated calls, or review the quota associated with the API key/project.

## 5. Poor RAG answers

Inspect retrieval independently:

```python
retriever.invoke("your question")
```

Then improve document quality, chunk sizes, metadata, the number of retrieved chunks, or the answer prompt.


# Key Takeaways

You have now practiced the main building blocks of a LangChain application:

**Prompt → Model → Parser → History → Tools → Embeddings → Retriever → RAG → Structured Application Output**

A sensible next progression is:

1. Add real PDF or web documents.
2. Add a persistent vector database.
3. Add LangSmith tracing and evaluation.
4. Add LangGraph for stateful multi-step agent workflows.
5. Build a Streamlit front end.
6. Deploy the application to a cloud platform.

---

## Official documentation used for this notebook

- LangChain Google Gemini chat-model integration
- LangChain Google Gemini embeddings integration
- LangChain structured-output documentation
- Google Gemini API model documentation

The notebook intentionally uses the current `langchain-google-genai` integration instead of older deprecated examples.
